# Evaluation 2 - Modelling Foundation

This notebook reconstructs and verifies the approved Evaluation 1 modelling population, recreates the fixed outer holdout, prepares the General, City, and Resort training scopes, validates the common grouped cross-validation structure, defines shared scoring conventions, and confirms that preprocessing is ready to be fitted inside future model pipelines.

**THIS NOTEBOOK DOES NOT TRAIN OR COMPARE ANY MACHINE-LEARNING ALGORITHM.**

Evaluation 2 algorithm-specific work begins only after this foundation is approved.

## 1. Imports and project root

The existing implementation in `src/preprocessing.py` is the authority for cleaning, feature engineering, exclusions, predictor grouping, and preprocessing construction. This notebook does not duplicate those rules.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.exceptions import NotFittedError
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.validation import check_is_fitted

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    root for root in root_candidates
    if (root / 'src' / 'preprocessing.py').exists() and (root / 'data' / 'raw').exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    CATEGORICAL_FEATURES,
    EXCLUDED_FEATURES,
    NUMERICAL_FEATURES,
    PREDICTOR_FEATURES,
    TARGET,
    build_preprocessor,
    create_predictor_groups,
    prepare_training_table,
    split_features_target,
)

RANDOM_STATE = 42
N_SPLITS = 5
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hotel_bookings.csv'
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data path: {RAW_PATH}')

Project root: d:\My GitHub Projects\hotel-cancellation
Raw data path: d:\My GitHub Projects\hotel-cancellation\data\raw\hotel_bookings.csv


## 2. Load the raw data

In [2]:
raw_df = pd.read_csv(RAW_PATH)
assert raw_df.shape == (119390, 32), f'Unexpected raw shape: {raw_df.shape}'
display(pd.DataFrame({
    'measure': ['Raw rows', 'Raw columns'],
    'value': [raw_df.shape[0], raw_df.shape[1]],
}))

,measure,value
0,Raw rows,119390
1,Raw columns,32


## 3. Recreate the approved modelling table

The source functions below apply the approved Evaluation 1 row policy, feature engineering, exclusions, and target separation.

In [3]:
training_table = prepare_training_table(raw_df)
X, y = split_features_target(training_table)

assert len(training_table) == 119210
assert X.shape == (119210, 30)
assert len(y) == 119210
assert TARGET not in X.columns
assert not set(EXCLUDED_FEATURES).intersection(X.columns)
assert set(X.columns) == set(PREDICTOR_FEATURES)
assert len(X.columns) == len(PREDICTOR_FEATURES)
assert set(NUMERICAL_FEATURES).isdisjoint(CATEGORICAL_FEATURES)
assert set(NUMERICAL_FEATURES).union(CATEGORICAL_FEATURES) == set(PREDICTOR_FEATURES)
assert len(NUMERICAL_FEATURES) == 20
assert len(CATEGORICAL_FEATURES) == 10
assert len(PREDICTOR_FEATURES) == 30
assert set(y.unique()).issubset({0, 1})

baseline_summary = pd.DataFrame({
    'measure': [
        'Raw rows', 'Rows removed', 'Modelling rows',
        'Unencoded predictors', 'Numerical predictors', 'Categorical predictors'
    ],
    'value': [len(raw_df), len(raw_df) - len(training_table), len(training_table),
              X.shape[1], len(NUMERICAL_FEATURES), len(CATEGORICAL_FEATURES)]
})
display(baseline_summary)

,measure,value
0,Raw rows,119390
1,Rows removed,180
2,Modelling rows,119210
3,Unencoded predictors,30
4,Numerical predictors,20
5,Categorical predictors,10


## Predictor-column contract

The approved predictor contract is the exact **set** of 30 named predictors, not the physical DataFrame column order. `X` currently preserves its source/preparation order and may begin with `hotel`.

`PREDICTOR_FEATURES` provides a canonical order when an operation specifically requires one. `create_predictor_groups()` already selects `X.loc[:, PREDICTOR_FEATURES]` for deterministic hashing, while `build_preprocessor()` selects columns by name through `ColumnTransformer`. Therefore, no source or preprocessing change is necessary.

## 4. Recreate predictor groups

The predictor-group hash is a partition helper built by the existing source function. It is not a model feature, booking ID, or customer ID.

In [4]:
predictor_groups = create_predictor_groups(X)
assert len(predictor_groups) == len(X)
assert predictor_groups.index.equals(X.index)
assert predictor_groups.notna().all()
assert 'predictor_group' not in X.columns
assert predictor_groups.nunique() == 83531

display(pd.DataFrame({
    'measure': ['Total modelling rows', 'Unique predictor groups'],
    'value': [len(X), predictor_groups.nunique()]
}))

,measure,value
0,Total modelling rows,119210
1,Unique predictor groups,83531


## 5. Recreate the fixed outer holdout

This is the approved outer split, not inner model-development CV. No alternate seed or fold is searched.

In [5]:
outer_splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
train_positions, test_positions = next(
    outer_splitter.split(X, y, groups=predictor_groups)
)

X_train_outer = X.iloc[train_positions].copy()
X_test_outer = X.iloc[test_positions].copy()
y_train_outer = y.iloc[train_positions].copy()
y_test_outer = y.iloc[test_positions].copy()
groups_train_outer = predictor_groups.iloc[train_positions].copy()
groups_test_outer = predictor_groups.iloc[test_positions].copy()

assert len(X_train_outer) == 95375
assert len(X_test_outer) == 23835
assert np.isclose(y.mean(), 0.37077, atol=0.0001)
assert np.isclose(y_train_outer.mean(), 0.37078, atol=0.0001)
assert np.isclose(y_test_outer.mean(), 0.37072, atol=0.0001)

outer_summary = pd.DataFrame({
    'scope': ['Overall', 'Outer train', 'Outer test'],
    'rows': [len(X), len(X_train_outer), len(X_test_outer)],
    'cancellation_rate': [y.mean(), y_train_outer.mean(), y_test_outer.mean()]
})
outer_summary['cancellation_rate'] = outer_summary['cancellation_rate'].mul(100).round(3).astype(str) + '%'
display(outer_summary)

,scope,rows,cancellation_rate
0,Overall,119210,37.077%
1,Outer train,95375,37.078%
2,Outer test,23835,37.072%


## 6. Verify outer group and complete-profile separation

In [6]:
outer_group_overlap = set(groups_train_outer.unique()).intersection(groups_test_outer.unique())
assert predictor_groups.nunique() == 83531
assert groups_train_outer.nunique() == 66831
assert groups_test_outer.nunique() == 16700
assert len(outer_group_overlap) == 0

def profile_index(frame):
    return pd.MultiIndex.from_frame(frame.loc[:, list(PREDICTOR_FEATURES)])

outer_profile_overlap = profile_index(X_train_outer).intersection(profile_index(X_test_outer))
assert len(outer_profile_overlap) == 0

outer_separation = pd.DataFrame({
    'measure': ['Total groups', 'Outer-train groups', 'Outer-test groups',
                'Group overlap', 'Identical complete predictor-profile overlap'],
    'value': [predictor_groups.nunique(), groups_train_outer.nunique(),
              groups_test_outer.nunique(), len(outer_group_overlap),
              len(outer_profile_overlap)]
})
display(outer_separation)

,measure,value
0,Total groups,83531
1,Outer-train groups,66831
2,Outer-test groups,16700
3,Group overlap,0
4,Identical complete predictor-profile overlap,0


## 7. Outer-test discipline

The objects `X_test_outer` and `y_test_outer` are reconstructed only to preserve and verify the approved holdout. They must not be used for model selection, hyperparameter tuning, tuning-range decisions, feature selection, or General-vs-specialist strategy decisions.

From this point onward, all model-development diagnostics in this foundation notebook use outer-training data only. No new algorithm/model results are calculated from the outer test set.

## 8. Prepare the three outer-training scopes

The specialist scopes retain the common 30-column schema, including `hotel`.

In [7]:
city_train_mask = X_train_outer['hotel'].eq('City Hotel')
resort_train_mask = X_train_outer['hotel'].eq('Resort Hotel')
city_test_mask = X_test_outer['hotel'].eq('City Hotel')
resort_test_mask = X_test_outer['hotel'].eq('Resort Hotel')
assert city_train_mask.sum() == 63044
assert resort_train_mask.sum() == 32331
assert city_test_mask.sum() == 16119
assert resort_test_mask.sum() == 7716

TRAINING_SCOPES = {
    'General': {
        'X': X_train_outer.copy(),
        'y': y_train_outer.copy(),
        'groups': groups_train_outer.copy(),
    },
    'City': {
        'X': X_train_outer.loc[city_train_mask].copy(),
        'y': y_train_outer.loc[city_train_mask].copy(),
        'groups': groups_train_outer.loc[city_train_mask].copy(),
    },
    'Resort': {
        'X': X_train_outer.loc[resort_train_mask].copy(),
        'y': y_train_outer.loc[resort_train_mask].copy(),
        'groups': groups_train_outer.loc[resort_train_mask].copy(),
    },
}

city_hotel_values = TRAINING_SCOPES['City']['X']['hotel'].dropna().unique()
resort_hotel_values = TRAINING_SCOPES['Resort']['X']['hotel'].dropna().unique()
assert len(city_hotel_values) == 1
assert city_hotel_values[0] == 'City Hotel'
assert len(resort_hotel_values) == 1
assert resort_hotel_values[0] == 'Resort Hotel'

for scope_name, scope in TRAINING_SCOPES.items():
    assert len(scope['X']) == len(scope['y']) == len(scope['groups'])
    assert scope['X'].index.equals(scope['y'].index)
    assert scope['X'].index.equals(scope['groups'].index)
    assert set(scope['X'].columns) == set(PREDICTOR_FEATURES)
    assert len(scope['X'].columns) == len(PREDICTOR_FEATURES)
    assert set(scope['y'].unique()).issubset({0, 1})
    assert 'predictor_group' not in scope['X'].columns

scope_summary = pd.DataFrame([
    {
        'Scope': name,
        'Rows': len(scope['X']),
        'Cancellation rate': scope['y'].mean(),
        'Unique predictor groups': scope['groups'].nunique(),
    }
    for name, scope in TRAINING_SCOPES.items()
])
scope_summary['Cancellation rate'] = scope_summary['Cancellation rate'].mul(100).round(3).astype(str) + '%'
display(scope_summary)

,Scope,Rows,Cancellation rate,Unique predictor groups
0,General,95375,37.078%,66831
1,City,63044,41.692%,40872
2,Resort,32331,28.081%,25959


The scope cancellation rates above are based only on outer-training data. The corresponding outer-test labels are not used for model-development analysis.

## 9. Define the common inner grouped CV

This is the inner model-development CV. It operates only on the relevant outer-training scope and is distinct from the fixed outer holdout. Every algorithm owner must use this same configuration.

In [8]:
def make_inner_cv():
    return StratifiedGroupKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
    )

display(Markdown('Inner CV: `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`'))

Inner CV: `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`

## 10. Validate inner CV for all scopes

No model is fitted in this section. The checks validate fold structure, alignment, group separation, validation coverage, and exact predictor-profile separation.

In [9]:
fold_diagnostics = []
fold_positions_by_scope = {}

for scope_name, scope in TRAINING_SCOPES.items():
    scope_X = scope['X']
    scope_y = scope['y']
    scope_groups = scope['groups']
    validation_counts = np.zeros(len(scope_X), dtype=int)
    scope_fold_positions = []

    for fold_number, (fold_train, fold_valid) in enumerate(
        make_inner_cv().split(scope_X, scope_y, groups=scope_groups), start=1
    ):
        validation_counts[fold_valid] += 1
        scope_fold_positions.append((fold_train.copy(), fold_valid.copy()))
        train_groups = set(scope_groups.iloc[fold_train])
        valid_groups = set(scope_groups.iloc[fold_valid])
        group_overlap = train_groups.intersection(valid_groups)
        assert len(group_overlap) == 0, f'{scope_name} fold {fold_number} group overlap'

        fold_train_X = scope_X.iloc[fold_train]
        fold_valid_X = scope_X.iloc[fold_valid]
        profile_overlap = profile_index(fold_train_X).intersection(profile_index(fold_valid_X))
        assert len(profile_overlap) == 0, f'{scope_name} fold {fold_number} profile overlap'

        fold_diagnostics.append({
            'scope': scope_name,
            'fold': fold_number,
            'training rows': len(fold_train),
            'validation rows': len(fold_valid),
            'training cancellation rate': scope_y.iloc[fold_train].mean(),
            'validation cancellation rate': scope_y.iloc[fold_valid].mean(),
            'training unique groups': len(train_groups),
            'validation unique groups': len(valid_groups),
            'group overlap': len(group_overlap),
            'profile overlap': len(profile_overlap),
        })

    assert np.all(validation_counts == 1)
    assert validation_counts.sum() == len(scope_X)
    fold_positions_by_scope[scope_name] = scope_fold_positions

fold_diagnostics_df = pd.DataFrame(fold_diagnostics)
display(fold_diagnostics_df)

,scope,fold,training rows,validation rows,training cancellation rate,validation cancellation rate,training unique groups,validation unique groups,group overlap,profile overlap
0,General,1,76315,19060,0.370805,0.370672,53463,13368,0,0
1,General,2,76315,19060,0.370805,0.370672,53459,13372,0,0
2,General,3,76314,19061,0.370810,0.370652,53457,13374,0,0
3,General,4,76242,19133,0.370675,0.371191,53473,13358,0,0
4,General,5,76314,19061,0.370797,0.370705,53472,13359,0,0
5,City,1,50454,12590,0.416954,0.416759,32696,8176,0,0
6,City,2,50453,12591,0.416943,0.416806,32711,8161,0,0
7,City,3,50455,12589,0.416946,0.416792,32688,8184,0,0
8,City,4,50361,12683,0.416791,0.417409,32703,8169,0,0
9,City,5,50453,12591,0.416943,0.416806,32690,8182,0,0


The exact predictor-profile checks are computationally reasonable here because the group definitions are based on the same complete final unencoded predictor profiles. The explicit group-overlap assertions remain the primary fold-separation control.

## 11. Reproducibility check

This verifies deterministic fold recreation for the current data order and configuration. It does not claim reproducibility across arbitrary library versions or reordered source data.

In [10]:
for scope_name, scope in TRAINING_SCOPES.items():
    recreated = list(make_inner_cv().split(scope['X'], scope['y'], groups=scope['groups']))
    original = fold_positions_by_scope[scope_name]
    assert len(original) == len(recreated) == N_SPLITS
    for (original_train, original_valid), (new_train, new_valid) in zip(original, recreated):
        np.testing.assert_array_equal(original_train, new_train)
        np.testing.assert_array_equal(original_valid, new_valid)

print('Reproducibility check passed for General, City, and Resort inner folds.')

Reproducibility check passed for General, City, and Resort inner folds.


## 12. Define common scoring conventions

Class `1` (Cancelled) is the positive class. F1 is the primary selection metric and Recall is an important secondary metric. The `roc_auc` scorer uses continuous model scores where the estimator supports them, rather than hard 0/1 predictions. The scoring dictionary is defined for later model notebooks only; it is not used to fit or evaluate a model here.

In [11]:
COMMON_SCORING = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, pos_label=1, zero_division=0),
    'recall': make_scorer(recall_score, pos_label=1, zero_division=0),
    'f1': make_scorer(f1_score, pos_label=1, zero_division=0),
    'roc_auc': 'roc_auc',
}
PRIMARY_SCORING = 'f1'
assert set(COMMON_SCORING) == {'accuracy', 'precision', 'recall', 'f1', 'roc_auc'}
assert PRIMARY_SCORING == 'f1'
display(pd.DataFrame({
    'scoring name': list(COMMON_SCORING),
    'primary metric': [name == PRIMARY_SCORING for name in COMMON_SCORING]
}))

,scoring name,primary metric
0,accuracy,False
1,precision,False
2,recall,False
3,f1,True
4,roc_auc,False


## 13. Verify preprocessor readiness without fitting

The existing `build_preprocessor` function constructs an unfitted transformer. Future algorithm notebooks must use a pipeline conceptually structured as:

```python
Pipeline([
    ('preprocessor', build_preprocessor(...)),
    ('classifier', algorithm),
])
```

This ensures that each inner training fold learns its own imputation statistics, scaling statistics, and encoding vocabularies. No classifier is instantiated in this notebook.

In [12]:
unfitted_preprocessor = build_preprocessor(TRAINING_SCOPES['General']['X'])
try:
    check_is_fitted(unfitted_preprocessor)
except NotFittedError:
    preprocessor_is_unfitted = True
else:
    preprocessor_is_unfitted = False

assert preprocessor_is_unfitted
print('Preprocessor readiness check passed: transformer remains unfitted.')

Preprocessor readiness check passed: transformer remains unfitted.


## 14. Foundation summary

In [13]:
summary_lines = [
    'Evaluation 1 baseline: 119,210 modelling rows; 30 unencoded predictors.',
    'Fixed outer holdout: 95,375 train; 23,835 test; group overlap = 0; identical predictor overlap = 0.',
    'Training scopes: General = 95,375; City = 63,044; Resort = 32,331.',
    'Inner validation: StratifiedGroupKFold, 5 folds, shuffle=True, random_state=42; zero group overlap in every fold.',
    'Common scoring: Accuracy, Precision, Recall, F1, ROC-AUC; primary = F1.',
    'Preprocessing: future fitting must occur inside the model Pipeline within each CV fold.',
    'No machine-learning algorithm has been trained in this notebook.',
]
display(Markdown('## Executed foundation summary\n\n' + '\n'.join(f'- {line}' for line in summary_lines)))

## Executed foundation summary

- Evaluation 1 baseline: 119,210 modelling rows; 30 unencoded predictors.
- Fixed outer holdout: 95,375 train; 23,835 test; group overlap = 0; identical predictor overlap = 0.
- Training scopes: General = 95,375; City = 63,044; Resort = 32,331.
- Inner validation: StratifiedGroupKFold, 5 folds, shuffle=True, random_state=42; zero group overlap in every fold.
- Common scoring: Accuracy, Precision, Recall, F1, ROC-AUC; primary = F1.
- Preprocessing: future fitting must occur inside the model Pipeline within each CV fold.
- No machine-learning algorithm has been trained in this notebook.